# Binaural: overt and masked onsets, intensity-level control

Part of the per-condition split of `cortical_analysis.ipynb`; runs on its own.
Shared settings (models, fit parameters, ROI, peak windows, plot window) are in
`parameters.py`.

**Results come from the cluster.** Every fit and model test is computed and kept on the
cluster; this notebook only reads them and saves nothing. In the cluster's JupyterHub
(any browser, laptop closed) it reads them from the cluster's disk; on the Mac (VS Code or
Jupyter), open the connection once in a terminal with `bash cluster/connect.sh`. The
`check_results` cell below asks the cluster for anything missing by itself, waits for the job
and then carries on, so one Run All goes from start to finish (keep the computer awake). See `cluster/README.md`.

**Overt and masked onsets** are made with each speaker at its level in the mixture that was played
(`predictors/inmix_onsets.py`; the speakers' own files are 4x louder, which made part of every
onset look masked). The split is tested against the same onsets unsplit (`FULL_INMIX`).

**Model comparisons:** TFCE permutation tests within the ROI found in the clean notebook
(`roi.txt`, read by the `check_results` cell; `TEST_REGION` in `parameters.py`), as
Brodbeck et al. 2020 did for the two-talker conditions.

**Peaks:** each subject's peak is searched within +-25 ms of where that condition's
group-average waveform peaks inside the search window (`add_anchored_peak`).

**Per-bin TRF plots** (`plot_trf_bins`): one line per frequency bin in its own colour
with a band of +-1 SEM across subjects (`BIN_ERROR` in `parameters.py`), the mean of
the bins as the thick black line, and the bins' centre frequencies on the right.

In [ ]:
from parameters import *

In [ ]:
# Checks that every result this notebook reads is on the cluster. If any are missing it
# asks the cluster for them, waits for the job (a line whenever its state changes) and
# carries on. Keep the computer awake while it waits.
check_results('binaural_overt_masked_onsets')

In [ ]:
# Your cluster jobs (PD = waiting for a slot, R = running).
cluster_queue()

In [ ]:
# The end of the latest cluster job's log (or cluster_log(<job number>)).
cluster_log()

## Binaural (overt onsets)

In [ ]:
# The ROI this notebook uses, imported from roi.txt: the clean-speech notebook found it
# from its onset and envelope test - baseline topomaps.
ROI = show_roi('binaural_overt_masked_onsets')

In [ ]:
# Overt/masked split vs. plain onsets (binaural)
# overt/masked = the split of each speaker's onsets into the part still visible in the mixture and
# the part the other speaker covers, with the speaker at its level in the mixture.
# Brodbeck et al. 2020 ran this comparison too: splitting onsets into overt/masked significantly improved the model overall, t = 6.81, p < .001.

# FULL_INMIX  = Eatt + Oatt + Eign + Oign + Emix + Omix   (Oatt, Oign at their level in the mixture)
# FULL_MASKED = Eatt + Oatt_overt + Oatt_masked + Eign + Oign_overt + Oign_masked + Emix + Omix
#
# Comparison: FULL_MASKED > FULL_INMIX
#   (does splitting Oatt/Oign into overt + masked improve the fit? overt + masked == Oatt/Oign,
#   so the two models carry the same onsets, only split or not)
#
# Full predictor names:
# FULL_MASKED (test model):
#   gammatone-8 + gammatone-on-overt-inmix-8 + gammatone-on-masked-inmix-8
#   + bg~gammatone-8 + bg~gammatone-on-overt-inmix-8 + bg~gammatone-on-masked-inmix-8
#   + mix~gammatone-8 + mix~gammatone-on-8
#
# FULL_INMIX (baseline model):
#   gammatone-8 + gammatone-on-inmix-8
#   + bg~gammatone-8 + bg~gammatone-on-inmix-8
#   + mix~gammatone-8 + mix~gammatone-on-8

data, diff, result = compute_direct_comparison(FULL_MASKED, FULL_INMIX, 'binaural')
show_topomap_trio(data, diff, result, 'Overt/masked split vs. plain onsets', 'binaural', None)

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt/masked split vs. plain onsets (binaural): ROI model comparison t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_swarm(diff, 'Overt/masked split vs. plain onsets', 'binaural')

In [ ]:
# Loaded once here, reused by every STRF/peak/waveform/attention-
# effect/masking-comparison cell in both this subsection and "masked
# onset" below - same reasoning as data_full above, for FULL_MASKED.
data_masked = load_trfs_saved(FULL_MASKED, 'binaural')

In [ ]:
# Overt onset attend unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: attended overt onsets unique contribution, t = 3.34, p = .027.
data, diff, result = compute_unique_contribution(FULL_MASKED, overt_onset(ATTEND), 'binaural')
show_topomap_trio(data, diff, result, 'Overt onset attend', 'binaural', overt_onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt onset attend (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_swarm(diff, 'Overt onset attend', 'binaural')

In [ ]:
# Overt onset ignore unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: ignored overt onsets unique contribution, t = 3.82, p = .016.
data, diff, result = compute_unique_contribution(FULL_MASKED, overt_onset(IGNORE), 'binaural')
show_topomap_trio(data, diff, result, 'Overt onset ignore', 'binaural', overt_onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt onset ignore (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_swarm(diff, 'Overt onset ignore', 'binaural')

In [ ]:
# Overt onset attend STRF (binaural)
strf = extract_strf(data_masked, 'gammatone_on_overt_inmix_8')
p = plot.Array(strf * 1e3, title='Overt onset attend STRF (binaural)', interpolation=STRF_INTERPOLATION, xlim=PLOT_WINDOW)

In [ ]:
# Overt onset ignore STRF (binaural)
strf = extract_strf(data_masked, 'bg_gammatone_on_overt_inmix_8')
p = plot.Array(strf * 1e3, title='Overt onset ignore STRF (binaural)', interpolation=STRF_INTERPOLATION, xlim=PLOT_WINDOW)

In [ ]:
# Overt onset attend peak amplitude & timing (binaural)
data = extract_trf(data_masked, 'gammatone_on_overt_inmix_8')
data = add_anchored_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_anchored_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Overt onset attend (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Overt onset attend (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
# Overt onset attend TRF per frequency bin (binaural)
p = plot_trf_bins(data_masked, 'gammatone_on_overt_inmix_8', 'Overt onset attend TRF (binaural)')

In [ ]:
# Overt onset ignore peak amplitude & timing (binaural)
data = extract_trf(data_masked, 'bg_gammatone_on_overt_inmix_8')
data = add_anchored_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_anchored_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Overt onset ignore (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Overt onset ignore (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
# Overt onset ignore TRF per frequency bin (binaural)
p = plot_trf_bins(data_masked, 'bg_gammatone_on_overt_inmix_8', 'Overt onset ignore TRF (binaural)')

In [ ]:
# Overt onset peak comparison across attend and ignore (binaural)
dss = []
for role_name, column in (('attend', 'gammatone_on_overt_inmix_8'), ('ignore', 'bg_gammatone_on_overt_inmix_8')):
    ds = extract_trf(data_masked, column)
    ds[:, 'role'] = role_name
    dss.append(ds)
data = combine(dss)
data = add_anchored_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max', by='role')
data = add_anchored_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min', by='role')

In [ ]:
# The overt onset TRF waveform attend/ignore were each just measured
# from, overlaid so their shape and relative timing are visible
# directly, not just as extracted peak numbers below.
p = plot.UTSStat('trf_sm*1e3', 'role', data=data, frame='t', h=2.5, w=4, xlim=PLOT_WINDOW,
                  title='Overt onset TRF: attend vs. ignore (binaural)', labels={'attend': 'Attend', 'ignore': 'Ignore'}, colors={'attend': 'red', 'ignore': 'blue'})

In [ ]:
print_peak_table(data, 'positive', 'role')
print_peak_table(data, 'negative', 'role')

In [ ]:
display(test.pairwise('positive_time', 'role', match='subject', data=data, corr=False, title='overt onset positive peak latency: attend vs. ignore'))
display(test.pairwise('negative_time', 'role', match='subject', data=data, corr=False, title='overt onset negative peak latency: attend vs. ignore'))

In [ ]:
p = plot.Barplot('positive_time', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='overt onset positive peak latency (binaural)')
p.set_xtick_rotation(30)
p = plot.Barplot('negative_time', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='overt onset negative peak latency (binaural)')
p.set_xtick_rotation(30)

In [ ]:
display(test.pairwise('positive_amp', 'role', match='subject', data=data, corr=False, title='overt onset positive peak amplitude: attend vs. ignore'))
display(test.pairwise('negative_amp', 'role', match='subject', data=data, corr=False, title='overt onset negative peak amplitude: attend vs. ignore'))

In [ ]:
p = plot.Barplot('positive_amp', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='overt onset positive peak amplitude (binaural)')
p.set_xtick_rotation(30)
p = plot.Barplot('negative_amp', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='overt onset negative peak amplitude (binaural)')
p.set_xtick_rotation(30)

In [ ]:
# Overt onset attention effect: attend - ignore (binaural)
p = plot_trf_bins(data_masked, 'gammatone_on_overt_inmix_8', 'Overt onset attention effect: attend - ignore (binaural)', subtract='bg_gammatone_on_overt_inmix_8')

## Binaural (masked onset)

In [ ]:
# Masked onset attend unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: attended masked onsets unique contribution, t = 8.42, p < .001 - the strongest of the four masking predictors.
data, diff, result = compute_unique_contribution(FULL_MASKED, masked_onset(ATTEND), 'binaural')
show_topomap_trio(data, diff, result, 'Masked onset attend', 'binaural', masked_onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked onset attend (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_swarm(diff, 'Masked onset attend', 'binaural')

In [ ]:
# Masked onset ignore unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: ignored masked onsets unique contribution, t = 5.23, p < .001 - the brain recovers masked information even for the ignored speaker.
data, diff, result = compute_unique_contribution(FULL_MASKED, masked_onset(IGNORE), 'binaural')
show_topomap_trio(data, diff, result, 'Masked onset ignore', 'binaural', masked_onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked onset ignore (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_swarm(diff, 'Masked onset ignore', 'binaural')

In [ ]:
# Masked onset attend STRF (binaural)
strf = extract_strf(data_masked, 'gammatone_on_masked_inmix_8')
p = plot.Array(strf * 1e3, title='Masked onset attend STRF (binaural)', interpolation=STRF_INTERPOLATION, xlim=PLOT_WINDOW)

In [ ]:
# Masked onset ignore STRF (binaural)
strf = extract_strf(data_masked, 'bg_gammatone_on_masked_inmix_8')
p = plot.Array(strf * 1e3, title='Masked onset ignore STRF (binaural)', interpolation=STRF_INTERPOLATION, xlim=PLOT_WINDOW)

In [ ]:
# Masked onset attend peak amplitude & timing (binaural)
data = extract_trf(data_masked, 'gammatone_on_masked_inmix_8')
data = add_anchored_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_anchored_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Masked onset attend (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Masked onset attend (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
# Masked onset attend TRF per frequency bin (binaural)
p = plot_trf_bins(data_masked, 'gammatone_on_masked_inmix_8', 'Masked onset attend TRF (binaural)')

In [ ]:
# Masked onset ignore peak amplitude & timing (binaural)
data = extract_trf(data_masked, 'bg_gammatone_on_masked_inmix_8')
data = add_anchored_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_anchored_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Masked onset ignore (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Masked onset ignore (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
# Masked onset ignore TRF per frequency bin (binaural)
p = plot_trf_bins(data_masked, 'bg_gammatone_on_masked_inmix_8', 'Masked onset ignore TRF (binaural)')

In [ ]:
# Masked onset peak comparison across attend and ignore (binaural)
dss = []
for role_name, column in (('attend', 'gammatone_on_masked_inmix_8'), ('ignore', 'bg_gammatone_on_masked_inmix_8')):
    ds = extract_trf(data_masked, column)
    ds[:, 'role'] = role_name
    dss.append(ds)
data = combine(dss)
data = add_anchored_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max', by='role')
data = add_anchored_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min', by='role')

In [ ]:
# The masked onset TRF waveform attend/ignore were each just measured
# from, overlaid so their shape and relative timing are visible
# directly, not just as extracted peak numbers below.
p = plot.UTSStat('trf_sm*1e3', 'role', data=data, frame='t', h=2.5, w=4, xlim=PLOT_WINDOW,
                  title='Masked onset TRF: attend vs. ignore (binaural)', labels={'attend': 'Attend', 'ignore': 'Ignore'}, colors={'attend': 'red', 'ignore': 'blue'})

In [ ]:
print_peak_table(data, 'positive', 'role')
print_peak_table(data, 'negative', 'role')

In [ ]:
display(test.pairwise('positive_time', 'role', match='subject', data=data, corr=False, title='masked onset positive peak latency: attend vs. ignore'))
display(test.pairwise('negative_time', 'role', match='subject', data=data, corr=False, title='masked onset negative peak latency: attend vs. ignore'))

In [ ]:
p = plot.Barplot('positive_time', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='masked onset positive peak latency (binaural)')
p.set_xtick_rotation(30)
p = plot.Barplot('negative_time', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='masked onset negative peak latency (binaural)')
p.set_xtick_rotation(30)

In [ ]:
display(test.pairwise('positive_amp', 'role', match='subject', data=data, corr=False, title='masked onset positive peak amplitude: attend vs. ignore'))
display(test.pairwise('negative_amp', 'role', match='subject', data=data, corr=False, title='masked onset negative peak amplitude: attend vs. ignore'))

In [ ]:
p = plot.Barplot('positive_amp', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='masked onset positive peak amplitude (binaural)')
p.set_xtick_rotation(30)
p = plot.Barplot('negative_amp', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='masked onset negative peak amplitude (binaural)')
p.set_xtick_rotation(30)

In [ ]:
# Masked onset attention effect: attend - ignore (binaural)
p = plot_trf_bins(data_masked, 'gammatone_on_masked_inmix_8', 'Masked onset attention effect: attend - ignore (binaural)', subtract='bg_gammatone_on_masked_inmix_8')

In [ ]:
# Overt vs. masked peak comparison, attend (binaural)
# Brodbeck et al. 2020 ran this comparison too: attended overt 72ms vs. attended masked 91ms (t=2.85,p=.009); combined (not stream-split) late negative peak, overt 133ms vs. masked 182ms (t=4.45,p<.001).
dss = []
for masking, column in (('overt', 'gammatone_on_overt_inmix_8'), ('masked', 'gammatone_on_masked_inmix_8')):
    ds = extract_trf(data_masked, column)
    ds[:, 'masking'] = masking
    dss.append(ds)
data = combine(dss)
data = add_anchored_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max', by='masking')
data = add_anchored_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min', by='masking')

In [ ]:
# Overt and masked onset TRFs of the attended speaker, overlaid
p = plot.UTSStat('trf_sm*1e3', 'masking', data=data, frame='t', h=2.5, w=4, xlim=PLOT_WINDOW,
                  title='Overt vs. masked onset TRF: attend (binaural)', labels=LABELS_MASKING, colors=COLORS_MASKING)

In [ ]:
print_peak_table(data, 'positive', 'masking')
print_peak_table(data, 'negative', 'masking')

In [ ]:
display(test.pairwise('positive_time', 'masking', match='subject', data=data, corr=False, title='attend positive peak latency: overt vs. masked'))
display(test.pairwise('negative_time', 'masking', match='subject', data=data, corr=False, title='attend negative peak latency: overt vs. masked'))

In [ ]:
display(test.pairwise('positive_amp', 'masking', match='subject', data=data, corr=False, title='attend positive peak amplitude: overt vs. masked'))
display(test.pairwise('negative_amp', 'masking', match='subject', data=data, corr=False, title='attend negative peak amplitude: overt vs. masked'))

In [ ]:
# Overt vs. masked peak comparison, ignore (binaural)
# Brodbeck et al. 2020 ran this comparison too: ignored overt 83ms vs. ignored masked 97ms (t=6.11,p<.001); combined (not stream-split) late negative peak, overt 133ms vs. masked 182ms (t=4.45,p<.001).
dss = []
for masking, column in (('overt', 'bg_gammatone_on_overt_inmix_8'), ('masked', 'bg_gammatone_on_masked_inmix_8')):
    ds = extract_trf(data_masked, column)
    ds[:, 'masking'] = masking
    dss.append(ds)
data = combine(dss)
data = add_anchored_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max', by='masking')
data = add_anchored_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min', by='masking')

In [ ]:
# Overt and masked onset TRFs of the ignored speaker, overlaid
p = plot.UTSStat('trf_sm*1e3', 'masking', data=data, frame='t', h=2.5, w=4, xlim=PLOT_WINDOW,
                  title='Overt vs. masked onset TRF: ignore (binaural)', labels=LABELS_MASKING, colors=COLORS_MASKING)

In [ ]:
print_peak_table(data, 'positive', 'masking')
print_peak_table(data, 'negative', 'masking')

In [ ]:
display(test.pairwise('positive_time', 'masking', match='subject', data=data, corr=False, title='ignore positive peak latency: overt vs. masked'))
display(test.pairwise('negative_time', 'masking', match='subject', data=data, corr=False, title='ignore negative peak latency: overt vs. masked'))

In [ ]:
display(test.pairwise('positive_amp', 'masking', match='subject', data=data, corr=False, title='ignore positive peak amplitude: overt vs. masked'))
display(test.pairwise('negative_amp', 'masking', match='subject', data=data, corr=False, title='ignore negative peak amplitude: overt vs. masked'))

In [ ]:
# Stream x masking interaction on peak amplitude (binaural)
# Brodbeck et al. 2020 ran this comparison too: stream x masking interaction, F(1,25) = 24.45, p < .001 - overt onsets show no early attention effect, masked onsets do.
dss = []
for stream_name, is_attend in (('attend', True), ('ignore', False)):
    for masking_name in ('overt', 'masked'):
        prefix = '' if is_attend else 'bg_'
        column = f'{prefix}gammatone_on_{masking_name}_8'
        ds = extract_trf(data_masked, column)
        ds[:, 'stream'] = stream_name
        ds[:, 'masking'] = masking_name
        dss.append(ds)
data = combine(dss)
data = add_anchored_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max', by=('stream', 'masking'))
display(test.ANOVA('positive_amp', 'stream * masking * subject', data=data, title='stream x masking interaction (binaural)'))

## Binaural (intensity-level control)

In [ ]:
# Masked-aware vs. level-aware (binaural)
# Brodbeck et al. 2020 ran this comparison too: the ignored-speaker-aware model still beat the loudness-only model, t = 9.21, p < .001 - the masking effect isn't just about volume.
data, diff, result = compute_direct_comparison(FULL_MASKED, LEVEL_AWARE, 'binaural')
show_topomap_trio(data, diff, result, 'Masked-aware vs. level-aware', 'binaural', None)

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked-aware vs. level-aware (binaural): ROI model comparison t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_swarm(diff, 'Masked-aware vs. level-aware', 'binaural')